In [2]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
import datetime as dt

In [3]:
# get daily data

df = pl.scan_ipc('crsp_daily.ftr').collect()

df

permno,caldt,siccd,prc,ret,vol,shr,excd,shrcd
i64,datetime[ns],i64,f64,f64,f64,f64,str,i64
10000,1986-01-07 00:00:00,3990,2.5625,null,1000.0,3680.0,"""Q""",10
10000,1986-01-08 00:00:00,3990,2.5,-0.02439,12800.0,3680.0,"""Q""",10
10000,1986-01-09 00:00:00,3990,2.5,0.0,1400.0,3680.0,"""Q""",10
10000,1986-01-10 00:00:00,3990,2.5,0.0,8500.0,3680.0,"""Q""",10
10000,1986-01-13 00:00:00,3990,2.625,0.05,5450.0,3680.0,"""Q""",10
…,…,…,…,…,…,…,…,…
93436,2026-01-26 00:00:00,3711,435.2,-0.030865,4.8764747e7,3.751e6,"""Q""",10
93436,2026-01-27 00:00:00,3711,430.9,-0.009881,3.7259749e7,3.751e6,"""Q""",10
93436,2026-01-28 00:00:00,3711,431.46,0.0013,5.4133715e7,3.751e6,"""Q""",10


In [4]:
# add log return col and remove unnecessary cols

df = (
    df
    .with_columns(
        pl.col('caldt').dt.date()
    )
    .with_columns(
        pl.col('ret').log1p().alias('logret')
    )
    .drop(['siccd','vol','shr','excd'])
)

df

permno,caldt,prc,ret,shrcd,logret
i64,date,f64,f64,i64,f64
10000,1986-01-07,2.5625,null,10,null
10000,1986-01-08,2.5,-0.02439,10,-0.024692
10000,1986-01-09,2.5,0.0,10,0.0
10000,1986-01-10,2.5,0.0,10,0.0
10000,1986-01-13,2.625,0.05,10,0.04879
…,…,…,…,…,…
93436,2026-01-26,435.2,-0.030865,10,-0.031351
93436,2026-01-27,430.9,-0.009881,10,-0.00993
93436,2026-01-28,431.46,0.0013,10,0.001299


In [5]:
# filter to dates from paper

df = df.filter((pl.col('caldt') >= dt.date(1965,1,1)) & (pl.col('caldt') <= dt.date(2002,12,31)))

df

permno,caldt,prc,ret,shrcd,logret
i64,date,f64,f64,i64,f64
10000,1986-01-07,2.5625,null,10,null
10000,1986-01-08,2.5,-0.02439,10,-0.024692
10000,1986-01-09,2.5,0.0,10,0.0
10000,1986-01-10,2.5,0.0,10,0.0
10000,1986-01-13,2.625,0.05,10,0.04879
…,…,…,…,…,…
93324,1985-12-03,0.109375,0.0,10,0.0
93324,1985-12-04,0.109375,0.0,10,0.0
93324,1985-12-05,0.109375,0.0,10,0.0


In [6]:
# create ret_21d monthly returns on daily basis for extension

df = (
    df
    .sort(['permno','caldt'])
    .with_columns(
        pl.col('logret').rolling_sum(window_size=21).over('permno').exp().sub(1).alias('ret_21d')
    )
)

# create ret_1m monthly returns for replication

df = (
    df
    .with_columns(
        pl.col('caldt').dt.truncate('1mo').alias('month')
    )
    .with_columns(
        pl.col('logret').sum().over(['permno','month']).exp().sub(1).alias('ret_1m')
    )
    .sort(['permno','month'])
)

df

permno,caldt,prc,ret,shrcd,logret,ret_21d,month,ret_1m
i64,date,f64,f64,i64,f64,f64,date,f64
10000,1986-01-07,2.5625,null,10,null,null,1986-01-01,0.707319
10000,1986-01-08,2.5,-0.02439,10,-0.024692,null,1986-01-01,0.707319
10000,1986-01-09,2.5,0.0,10,0.0,null,1986-01-01,0.707319
10000,1986-01-10,2.5,0.0,10,0.0,null,1986-01-01,0.707319
10000,1986-01-13,2.625,0.05,10,0.04879,null,1986-01-01,0.707319
…,…,…,…,…,…,…,…,…
93324,1985-12-03,0.109375,0.0,10,0.0,4.5238e-7,1985-12-01,0.166667
93324,1985-12-04,0.109375,0.0,10,0.0,4.5238e-7,1985-12-01,0.166667
93324,1985-12-05,0.109375,0.0,10,0.0,0.166667,1985-12-01,0.166667
